# 1. Carga de datos

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

esqueleto = pd.read_parquet('/content/drive/MyDrive/TFM/favorita_panel_corregido.parquet')
print(esqueleto.shape)

Mounted at /content/drive
(4804366, 22)


# 2. Preparación de variables

In [2]:
if 'type' in esqueleto.columns:
    esqueleto = esqueleto.rename(columns={'type': 'store_type'})

for col in ['store_nbr', 'item_nbr', 'city', 'state', 'store_type', 'cluster', 'class']:
    esqueleto[col] = esqueleto[col].astype(str)

# 3. Partición de datos

In [3]:
fecha_max = esqueleto['date'].max()
test_inicio = fecha_max - pd.Timedelta(days=60)
val_inicio = test_inicio - pd.Timedelta(days=60)

train = esqueleto[esqueleto['date'] < val_inicio]
val = esqueleto[(esqueleto['date'] >= val_inicio) & (esqueleto['date'] < test_inicio)]

print('Train:', train.shape, '| Val:', val.shape)

Train: (4322665, 22) | Val: (238860, 22)


# 4. Instalación e importaciones de PyTorch Forecasting

In [4]:
!pip install pytorch-forecasting pytorch-lightning lightning -q

from pytorch_forecasting import TimeSeriesDataSet, DeepAR
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import NegativeBinomialDistributionLoss
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
import torch

print(torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 63.8 MB/s eta 0:00:00
True


# 5. Ponderación por volumen

In [5]:
volumen_por_serie = esqueleto.groupby(['store_nbr', 'item_nbr'])['unit_sales'].transform('mean')
esqueleto['peso_muestra'] = 1 / np.sqrt(volumen_por_serie + 0.1)
esqueleto['peso_muestra'] = esqueleto['peso_muestra'] / esqueleto['peso_muestra'].mean()

# 6. Reconstrucción de TimeSeriesDataSet corregido

In [6]:
max_encoder_length = 90
max_prediction_length = 30
training_cutoff = train['time_idx'].max()

training = TimeSeriesDataSet(
    esqueleto[esqueleto.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="unit_sales",
    group_ids=["store_nbr", "item_nbr"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["city", "state", "store_type", "cluster", "class"],
    time_varying_known_reals=["time_idx", "year", "month", "day_of_week", "is_weekend",
                               "es_feriado", "onpromotion", "edad"],
    time_varying_unknown_reals=["unit_sales"],
    target_normalizer=GroupNormalizer(groups=["store_nbr", "item_nbr"], center=False),
    weight="peso_muestra",
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=False,
)

validation = TimeSeriesDataSet.from_dataset(
    training, esqueleto, min_prediction_idx=training_cutoff + 1, stop_randomization=True
)

/usr/local/lib/python3.13/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 88 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__store_nbr': '1', '__group_id__item_nbr': '1428779'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2002136'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2027777'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2027827'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053590'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053610'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053614'}, {'__group_id__store_nbr': '10', '__group_id__item_nbr': '2033805'}, {'__group_id__store_nbr': '10', '__group_id__item_nbr': '2053590'}, {'__group_id__store_nbr': '1

# 7. Dataloaders

In [7]:
batch_size = 128
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

# 8. DeepAR - V8
### 8.1. Segunda configuración (hidden_size=64, batches=200), con binomial negativa

In [8]:
deepar_v8 = DeepAR.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=64,
    rnn_layers=2,
    loss=NegativeBinomialDistributionLoss(),
    logging_metrics=torch.nn.ModuleList([]),  # ajuste para resolver error de anterior notebook
)

trainer_v8 = pl.Trainer(
    max_epochs=30, accelerator="auto", enable_model_summary=True,
    callbacks=[EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=5, mode="min")],
    gradient_clip_val=0.1, limit_train_batches=200, limit_val_batches=50,
)
trainer_v8.fit(deepar_v8, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
trainer_v8.save_checkpoint('/content/drive/MyDrive/TFM/deepar_v8_corregido.ckpt')

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                   ┃ Type                             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                   │ NegativeBinomialDistributionLoss │      0 │ train │     0 │
│ 1 │ logging_metrics        │ ModuleList                       │      0 │ train │     0 │
│ 2 │ embeddings             │ MultiEmbedding                   │    273 │ train │     0 │
│ 3 │ rnn                    │ LSTM                             │ 58.6 K │ train │     0 │
│ 4 │ distribution_projector │ Linear                           │    130 │ train │     0 │
└───┴────────────────────────┴──────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 59.0 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 59.0 K                                                                                               
Total estimated model params size (MB): 0.236                                                                      
Modules in train mode: 11                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `weights_only` was not set, defaulting to `False`.
INFO:lightning.pytorch.trainer.connectors.checkpoint_connector:`weights_only` was not set, defaulting to `False`.
